# Swim relay problem (Van Roy and Mason)
The coach of a swim team needs to assign four of five swimmers to a 200-yard medley relay, exactly one per stroke and each swimmer at most once. One swimmer is unused. The swimmers are good in more than one stroke, so it's not clear which swimmer to assign to which stroke. Here are their best times in seconds per 50-yard leg:

|Stroke       | Carl | Chris | David | Tony | Ken  |
|-------------|:----:|:-----:|:-----:|:----:|:----:|
|Backstroke   | 37.7 | 32.9  | 33.8  | 37.0 | 35.4 |
|Breaststroke | 43.4 | 33.1  | 42.2  | 34.7 | 41.8 |
|Butterfly    | 33.3 | 28.5  | 38.9  | 30.4 | 33.6 |
|Freestyle    | 29.2 | 26.4  | 29.6  | 28.5 | 31.1 |

Open the course repository root in VS Code, select the Julia 1.12 kernel with the course environment, and choose **Run All**. All required packages are already included, and no external data files are needed.


### Solution

With five swimmers and four strokes, each swimmer's assignment constraint is at most one ($\leq 1$), while each stroke's constraint is exactly one ($=1$).

The continuous assignment LP has an integral optimal basic solution because it is a network-flow model with integer supplies and demands. Binary variables are not needed for this example.


In [ ]:
using JuMP, HiGHS, NamedArrays
import MathOptInterface as MOI

# Types of strokes
strokes = [ :backstroke, :breaststroke, :butterfly, :freestyle ]

# Names of the swimmers
names = [ :Carl, :Chris, :David, :Tony, :Ken ]

# Time of each swimmer for each stroke
raw = [ 37.7 32.9 33.8 37.0 35.4
        43.4 33.1 42.2 34.7 41.8
        33.3 28.5 38.9 30.4 33.6
        29.2 26.4 29.6 28.5 31.1 ]

times = NamedArray( raw, (strokes,names), ("stroke","name"))

m = Model(HiGHS.Optimizer)
set_silent(m)

@variable(m, x[strokes,names] >= 0)

# each swimmer swims at most one event
@constraint(m, a[j in names], sum(x[i,j] for i in strokes) <= 1 )

# each event has exactly one swimmer
@constraint(m, b[i in strokes], sum(x[i,j] for j in names) == 1 )

@objective(m, Min, sum( x[i,j]*times[i,j] for i in strokes, j in names ) )

optimize!(m)
if termination_status(m) != MOI.OPTIMAL || primal_status(m) != MOI.FEASIBLE_POINT
    error("No optimal primal solution: $(termination_status(m)), $(primal_status(m))")
end

assignment = NamedArray( [ (value.(x[i,j])) for i in strokes, j in names ], (strokes, names), ("stroke","name"))

### Balanced minimum-cost network-flow version

Add a dummy stroke, `:unused`, with demand 1 and zero assignment cost. Each of the five swimmers supplies 1 unit; each real stroke and the dummy stroke demands 1 unit. Total supply and demand are both 5, so every node can use an equality balance constraint.

Arcs go from swimmers to strokes. The swimmer assigned to `:unused` does not swim. Real-arc costs are the times from the table above, and dummy-arc costs are zero. The data and model are in separate cells; the model cell is copied unchanged from [12-mcnf.ipynb](12-mcnf.ipynb).


In [ ]:
# Reuse names, strokes, and times from the original instance above.
assignment_strokes = [strokes; :unused]
nodes = [names; assignment_strokes]
arcs = [(i,j) for i in names for j in assignment_strokes]

# Real assignments cost their swim time; the dummy assignment costs zero.
c = Dict((i,j) => (j == :unused ? 0.0 : times[j,i]) for (i,j) in arcs)

# One unit of supply per swimmer and one unit of demand per stroke.
b = Dict(i => (i in names ? 1 : -1) for i in nodes)

# Node balance already limits each assignment to at most one unit.
ℓ = Dict((i,j) => 0 for (i,j) in arcs)
u = Dict((i,j) => Inf for (i,j) in arcs)


#### Model


In [ ]:
using JuMP, HiGHS
import MathOptInterface as MOI

m = Model()

@variable(m, ℓ[i,j] <= x[(i,j) in arcs] <= u[i,j])

@objective(m, Min,
    sum(c[i,j]x[(i,j)] for (i,j) in arcs))

@constraint(m, flowbalance[i in nodes],
    sum(x[(i,j)] for j in nodes if (i,j) in arcs) -
    sum(x[(j,i)] for j in nodes if (j,i) in arcs)
    == b[i])

set_optimizer(m, HiGHS.Optimizer)
set_silent(m)
optimize!(m)


#### Solution


In [ ]:
println("Status: ", termination_status(m))
if termination_status(m) != MOI.OPTIMAL || primal_status(m) != MOI.FEASIBLE_POINT
    error("No optimal primal solution: $(termination_status(m)), $(primal_status(m))")
end

println("Minimum total relay time = ", round(objective_value(m), digits=1), " seconds")
for (i,j) in arcs
    if value(x[(i,j)]) > 0.00001
        if j == :unused
            println("Unused swimmer: ", i)
        else
            println(i, " swims ", j, ": ", times[j,i], " seconds")
        end
    end
end
